# Task 1: News Topic Classifier Using BERT

**Objective:** Fine-tune a transformer model (`bert-base-uncased`) to classify news headlines into topic categories.

**Dataset:** AG News (World, Sports, Business, Sci/Tech)

**Pipeline:**
1. Load & preprocess the dataset
2. Tokenize with the BERT tokenizer
3. Fine-tune `bert-base-uncased` with Hugging Face `Transformers`
4. Evaluate with accuracy & F1-score
5. Deploy the model with a simple **Gradio** app for live interaction

> **Note on the dataset file:** This notebook ships with `ag_news_sample.csv`, a small, balanced, 600-row sample (150 per class) in the same 4 AG News categories, so the whole notebook runs end-to-end without needing internet access to the Hugging Face Hub. For a full-scale run, the notebook also includes a one-line switch (`USE_FULL_HF_DATASET = True`) to load the complete ~120k-row official AG News dataset directly from Hugging Face `datasets` — use that for your real submission if you have internet/Hub access.


## 1. Install dependencies

In [1]:
# Uncomment if running in a fresh environment (e.g. Google Colab)
# !pip install -q transformers datasets evaluate scikit-learn torch gradio accelerate


In [2]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

torch.manual_seed(42)
np.random.seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


Using device: cpu


## 2. Load the dataset

Set `USE_FULL_HF_DATASET = True` below if you have internet access and want to train on the
full, official AG News dataset (~120,000 train / 7,600 test rows) via the Hugging Face `datasets`
library. Otherwise the notebook uses the bundled `ag_news_sample.csv`.


In [3]:
USE_FULL_HF_DATASET = False  # set True to pull the full official AG News dataset from HF Hub

label_names = ["World", "Sports", "Business", "Sci/Tech"]

if USE_FULL_HF_DATASET:
    from datasets import load_dataset
    raw = load_dataset("ag_news")
    train_df = pd.DataFrame(raw["train"])
    test_df = pd.DataFrame(raw["test"])
    train_df.rename(columns={"text": "text", "label": "label_num"}, inplace=True)
    test_df.rename(columns={"text": "text", "label": "label_num"}, inplace=True)
else:
    df = pd.read_csv("ag_news_sample.csv")
    # Combine title + description into a single text field, matching AG News's own format
    df["text"] = df["title"] + " " + df["description"]
    train_df, test_df = train_test_split(
        df, test_size=0.2, random_state=42, stratify=df["label_num"]
    )

print("Train size:", len(train_df))
print("Test size:", len(test_df))
train_df.head()


Train size: 480
Test size: 120


,title,description,label,label_num,text
98,Australia: The president hosts international s...,Australia the president hosts international su...,World,0,Australia: The president hosts international s...
318,An artificial intelligence firm test new mater...,An artificial intelligence firm test new mater...,Sci/Tech,3,An artificial intelligence firm test new mater...
218,Egypt: The ruling party faces criticism over h...,Egypt the ruling party faces criticism over ha...,World,0,Egypt: The ruling party faces criticism over h...
391,the Riverside Warriors secures qualification f...,"In the marathon organizing committee, the rive...",Sports,1,the Riverside Warriors secures qualification f...
447,Egypt: The ruling party announces new trade ag...,Egypt the ruling party announces new trade agr...,World,0,Egypt: The ruling party announces new trade ag...


## 3. Preprocess & tokenize the dataset

We use `BertTokenizerFast` (`bert-base-uncased`) to tokenize headlines, with padding/truncation
to a fixed max length, then wrap everything in a Hugging Face `Dataset` object.


In [4]:
from transformers import BertTokenizerFast
from datasets import Dataset

MODEL_NAME = "bert-base-uncased"
MAX_LEN = 64

tokenizer = BertTokenizerFast.from_pretrained(MODEL_NAME)

train_ds = Dataset.from_pandas(train_df[["text", "label_num"]].rename(columns={"label_num": "label"}).reset_index(drop=True))
test_ds = Dataset.from_pandas(test_df[["text", "label_num"]].rename(columns={"label_num": "label"}).reset_index(drop=True))

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN,
    )

train_ds = train_ds.map(tokenize_batch, batched=True)
test_ds = test_ds.map(tokenize_batch, batched=True)

train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

train_ds, test_ds


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

C:\Users\MS\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\MS\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/480 [00:00<?, ? examples/s]

Map:   0%|          | 0/120 [00:00<?, ? examples/s]

(Dataset({
     features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
     num_rows: 480
 }),
 Dataset({
     features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
     num_rows: 120
 }))

## 4. Load `bert-base-uncased` for sequence classification

In [5]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_names),
    id2label={i: name for i, name in enumerate(label_names)},
    label2id={name: i for i, name in enumerate(label_names)},
).to(device)


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 5. Define evaluation metrics (accuracy & F1-score)

In [6]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    f1_macro = f1_score(labels, preds, average="macro")
    f1_weighted = f1_score(labels, preds, average="weighted")
    return {
        "accuracy": acc,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted,
    }


## 6. Fine-tune the model

Uses Hugging Face `Trainer` for the training loop. Adjust `num_train_epochs` and
`per_device_train_batch_size` depending on your hardware (GPU strongly recommended for the
full dataset).


In [7]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./bert-news-classifier",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=20,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics,
)

trainer.train()


C:\Users\MS\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
1,0.847169,0.244159,1.000000,1.000000,1.000000
2,0.115651,0.053963,1.000000,1.000000,1.000000
3,0.058764,0.033623,1.000000,1.000000,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\MS\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\MS\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=90, training_loss=0.30119063589308, metrics={'train_runtime': 1693.6531, 'train_samples_per_second': 0.85, 'train_steps_per_second': 0.053, 'total_flos': 47360840417280.0, 'train_loss': 0.30119063589308, 'epoch': 3.0})

## 7. Evaluate the fine-tuned model

In [8]:
eval_results = trainer.evaluate()
print(eval_results)


C:\Users\MS\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro,F1 Weighted
0.058764,0.243173,3,1.000000,1.000000,1.000000


{'eval_loss': 0.2431734949350357, 'eval_accuracy': 1.0, 'eval_f1_macro': 1.0, 'eval_f1_weighted': 1.0}


In [9]:
# Detailed per-class report
predictions = trainer.predict(test_ds)
y_pred = np.argmax(predictions.predictions, axis=-1)
y_true = predictions.label_ids

print(classification_report(y_true, y_pred, target_names=label_names))


C:\Users\MS\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


              precision    recall  f1-score   support

       World       1.00      1.00      1.00        30
      Sports       1.00      1.00      1.00        30
    Business       1.00      1.00      1.00        30
    Sci/Tech       1.00      1.00      1.00        30

    accuracy                           1.00       120
   macro avg       1.00      1.00      1.00       120
weighted avg       1.00      1.00      1.00       120



## 8. Save the fine-tuned model & tokenizer

In [10]:
SAVE_DIR = "./bert-news-classifier-final"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("Model saved to", SAVE_DIR)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./bert-news-classifier-final


## 9. Quick inference helper

In [11]:
def predict_topic(text, model=model, tokenizer=tokenizer, device=device):
    model.eval()
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=MAX_LEN).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()[0]
    pred_idx = int(np.argmax(probs))
    return label_names[pred_idx], {label_names[i]: float(probs[i]) for i in range(len(label_names))}

# Example
sample_headline = "Central bank raises interest rates amid inflation fears"
label, probs = predict_topic(sample_headline)
print("Predicted topic:", label)
print("Probabilities:", probs)


Predicted topic: Business
Probabilities: {'World': 0.3028270900249481, 'Sports': 0.18695352971553802, 'Business': 0.33388751745224, 'Sci/Tech': 0.17633190751075745}


## 10. Deploy with Gradio for live interaction

Run this cell to launch a simple web UI where you can type a headline and get the predicted topic.
Set `share=True` if you want a public link (e.g. when running in Colab).


In [12]:
import gradio as gr

def classify_headline(headline):
    label, probs = predict_topic(headline)
    return label, probs

demo = gr.Interface(
    fn=classify_headline,
    inputs=gr.Textbox(lines=2, placeholder="Enter a news headline..."),
    outputs=[gr.Label(label="Predicted Topic"), gr.Label(label="Class Probabilities", num_top_classes=4)],
    title="News Topic Classifier (BERT)",
    description="Fine-tuned bert-base-uncased model classifying headlines into World, Sports, Business, or Sci/Tech.",
    examples=[
        "Local team wins championship in overtime thriller",
        "Tech giant unveils new AI-powered chip",
        "Stock markets rally after central bank announcement",
        "Diplomats meet to discuss regional peace agreement",
    ],
)

demo.launch(share=True)


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


<!-- ### Alternative: Streamlit app

If you prefer Streamlit over Gradio, save the following as `app.py` and run `streamlit run app.py`:

```python
import streamlit as st
import torch
from transformers import BertTokenizerFast, BertForSequenceClassification

MODEL_DIR = "./bert-news-classifier-final"
label_names = ["World", "Sports", "Business", "Sci/Tech"]

@st.cache_resource
def load_model():
    tokenizer = BertTokenizerFast.from_pretrained(MODEL_DIR)
    model = BertForSequenceClassification.from_pretrained(MODEL_DIR)
    model.eval()
    return tokenizer, model

tokenizer, model = load_model()

st.title("News Topic Classifier (BERT)")
headline = st.text_area("Enter a news headline:")

if st.button("Classify") and headline:
    inputs = tokenizer(headline, return_tensors="pt", truncation=True, padding=True, max_length=64)
    with torch.no_grad():
        outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=-1)[0]
    pred_idx = int(torch.argmax(probs))
    st.subheader(f"Predicted Topic: {label_names[pred_idx]}")
    st.bar_chart({label_names[i]: float(probs[i]) for i in range(len(label_names))})
``` -->
